# CV R² comparison: standard vs session-shift aPRF

Does allowing the preferred value (mode) to shift between sessions improve
cross-validated prediction? The session-shift model has one extra free
parameter per voxel (mode_2), so it should only win if the tuning genuinely
changes between conditions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.maskers import NiftiMasker
from nilearn import image as nli

from abstract_values.utils.data import Subject, BIDS_FOLDER

sns.set_theme(style='ticks', font_scale=1.1)

In [ ]:
SMOOTHED = False
smooth_label = '_smoothed' if SMOOTHED else ''

ROIS = [
    ('NPCr', None),
    ('BensonV1', 'LR'),
]

# Auto-discover subjects that have BOTH standard and shift CV R²
std_cv_dir  = BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'aprf.cv'
shift_cv_dir = BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'aprf-shift.cv'

SUBJECTS = []
for d in sorted(std_cv_dir.iterdir()) if std_cv_dir.exists() else []:
    if not d.is_dir() or not d.name.startswith('sub-'):
        continue
    sub_id = d.name.removeprefix('sub-')
    std_fn = d / 'func' / f'{d.name}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz'
    shift_fn = shift_cv_dir / d.name / 'func' / f'{d.name}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz'
    if std_fn.exists() and shift_fn.exists():
        SUBJECTS.append(sub_id)

print(f'Subjects with both standard and shift CV R²: {SUBJECTS}')

In [ ]:
data = {}

for sub_id in SUBJECTS:
    sub = Subject(sub_id, bids_folder=BIDS_FOLDER)

    std_fn = (std_cv_dir / f'sub-{sub_id}' / 'func' /
              f'sub-{sub_id}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz')
    shift_fn = (shift_cv_dir / f'sub-{sub_id}' / 'func' /
                f'sub-{sub_id}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz')

    ref = nli.load_img(str(std_fn))

    roi_masks = {}
    for roi_name, hemi in ROIS:
        try:
            roi_masks[roi_name] = sub.get_roi_mask(roi_name, hemi=hemi)
        except FileNotFoundError:
            print(f'  sub-{sub_id}: no {roi_name} mask, skipping ROI')

    for roi_name, mask_img in roi_masks.items():
        masker = NiftiMasker(mask_img=mask_img,
                             target_affine=ref.affine,
                             target_shape=ref.shape[:3]).fit()
        cvr2_std   = masker.transform(str(std_fn)).squeeze()
        cvr2_shift = masker.transform(str(shift_fn)).squeeze()

        data[(sub_id, roi_name)] = dict(
            cvr2_std=cvr2_std, cvr2_shift=cvr2_shift)

        n = len(cvr2_std)
        better = (cvr2_shift > cvr2_std).sum()
        print(f'sub-{sub_id} {roi_name}: {n} voxels, '
              f'shift > standard in {better}/{n} ({better/n*100:.1f}%), '
              f'mean diff = {(cvr2_shift - cvr2_std).mean():.4f}')

## Scatter: standard vs session-shift CV R²

In [ ]:
if data:
    roi_names = sorted({roi for _, roi in data.keys()})
    fig, axes = plt.subplots(len(SUBJECTS), len(roi_names),
                              figsize=(5 * len(roi_names), 5 * len(SUBJECTS)),
                              squeeze=False, constrained_layout=True)

    for i, sub_id in enumerate(SUBJECTS):
        for j, roi_name in enumerate(roi_names):
            ax = axes[i, j]
            key = (sub_id, roi_name)
            if key not in data:
                ax.set_visible(False)
                continue

            d = data[key]
            ax.scatter(d['cvr2_std'], d['cvr2_shift'],
                       s=2, alpha=0.3, linewidths=0, c='steelblue')
            lims = [min(d['cvr2_std'].min(), d['cvr2_shift'].min()),
                    max(d['cvr2_std'].max(), d['cvr2_shift'].max())]
            ax.plot(lims, lims, 'k--', lw=1, alpha=0.5)
            ax.set_xlabel('CV R² (standard)')
            ax.set_ylabel('CV R² (session-shift)')
            ax.set_title(f'sub-{sub_id}  {roi_name}')
            ax.set_aspect('equal')

            # Annotate
            diff = d['cvr2_shift'] - d['cvr2_std']
            ax.text(0.05, 0.95,
                    f'mean diff={diff.mean():.4f}\n'
                    f'shift > std: {(diff>0).sum()}/{len(diff)}',
                    transform=ax.transAxes, fontsize=8, va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    fig.suptitle('Cross-validated R²: standard vs session-shift aPRF',
                 fontsize=13)
    plt.show()

## Distribution of CV R² difference (shift - standard)

In [ ]:
if data:
    roi_names = sorted({roi for _, roi in data.keys()})
    fig, axes = plt.subplots(len(SUBJECTS), len(roi_names),
                              figsize=(5 * len(roi_names), 3.5 * len(SUBJECTS)),
                              squeeze=False, constrained_layout=True)

    for i, sub_id in enumerate(SUBJECTS):
        for j, roi_name in enumerate(roi_names):
            ax = axes[i, j]
            key = (sub_id, roi_name)
            if key not in data:
                ax.set_visible(False)
                continue

            d = data[key]
            diff = d['cvr2_shift'] - d['cvr2_std']
            ax.hist(diff, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
            ax.axvline(0, color='k', lw=1.5, ls='--')
            ax.axvline(diff.mean(), color='tomato', lw=2,
                       label=f'mean={diff.mean():.4f}')
            ax.axvline(np.median(diff), color='orange', lw=2, ls=':',
                       label=f'median={np.median(diff):.4f}')
            ax.set_xlabel('CV R² (shift) - CV R² (standard)')
            ax.set_ylabel('voxel count')
            ax.set_title(f'sub-{sub_id}  {roi_name}')
            ax.legend(fontsize=8)

    fig.suptitle('CV R² improvement from session-shift model', fontsize=13)
    sns.despine()
    plt.show()

## Summary across participants

In [ ]:
if data:
    rows = []
    for (sub_id, roi_name), d in data.items():
        diff = d['cvr2_shift'] - d['cvr2_std']
        rows.append(dict(
            subject=sub_id, roi=roi_name,
            mean_cvr2_std=d['cvr2_std'].mean(),
            mean_cvr2_shift=d['cvr2_shift'].mean(),
            mean_diff=diff.mean(),
            median_diff=np.median(diff),
            pct_shift_better=(diff > 0).mean() * 100,
            n_voxels=len(diff),
        ))

    summary = pd.DataFrame(rows)
    print(summary.round(4).to_string(index=False))

    # Swarmplot of mean CV R² difference
    if summary['subject'].nunique() > 1:
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.swarmplot(data=summary, x='roi', y='mean_diff', size=8,
                      color='0.3', ax=ax)
        ax.axhline(0, color='k', lw=1, ls='--', alpha=0.5)
        ax.set_xlabel('')
        ax.set_ylabel('Mean CV R² (shift - standard)')
        ax.set_title('Does the session-shift model improve prediction?')
        sns.despine()
        plt.tight_layout()
        plt.show()